In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
train_df= pd.read_pickle("train_final.pkl")
val_df= pd.read_pickle("val_final.pkl")

print(train_df.shape, val_df.shape)
print(train_df.columns)

(1971, 15) (500, 15)
Index(['review_id', 'review_text', 'star_rating', 'business_category',
       'platform', 'aspects', 'aspect_sentiments', 'clean_text', 'len_before',
       'len_after', 'aspects_parsed', 'sentiments_parsed', 'asp_labels',
       'sent_labels', 'model_input'],
      dtype='object')


In [5]:
def create_label_string(row):
    """
    Create labels from aspects and sentiments
    """
    aspects = row.get('aspects_parsed', [])
    sentiments = row.get('sentiments_parsed', [])
    
    # Handle different data types
    if hasattr(aspects, 'tolist'):
        aspects = aspects.tolist()
    if hasattr(sentiments, 'tolist'):
        sentiments = sentiments.tolist()
    
    if not aspects or not sentiments:
        return []
    
    labels = []
    for aspect, sentiment in zip(aspects, sentiments):
        if aspect and sentiment and aspect != 'none' and sentiment != 'none':
            labels.append(f"{aspect}: {sentiment}")
    
    return labels

# Test on first row
print("Testing label creation...")
print(f"Row 0 labels: {create_label_string(train_df.iloc[0])}")

Testing label creation...
Row 0 labels: ['app_experience: app_experience', 'delivery: delivery']


In [6]:
print("Preparing data...")

# Create labels
train_df['label_list'] = train_df.apply(create_label_string, axis=1)
val_df['label_list'] = val_df.apply(create_label_string, axis=1)

# Create input text (use clean_text if available)
train_df['input_text'] = train_df['clean_text'].fillna(train_df['review_text'])
val_df['input_text'] = val_df['clean_text'].fillna(val_df['review_text'])

# Remove rows with empty text
train_df = train_df.dropna(subset=['input_text'])
val_df = val_df.dropna(subset=['input_text'])

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# Show sample
print("\nSample:")
print(f"Text: {train_df.iloc[0]['input_text'][:100]}...")
print(f"Labels: {train_df.iloc[0]['label_list']}")

Preparing data...
Train samples: 1971
Validation samples: 500

Sample:
Text: لا يوجد الدفع بالبطاقه عند الاستلام...
Labels: ['app_experience: app_experience', 'delivery: delivery']


In [7]:
# Get all unique labels
all_labels = set()
for labels in train_df['label_list']:
    for label in labels:
        all_labels.add(label)

unique_labels = sorted(list(all_labels))
print(f"Total unique labels: {len(unique_labels)}")
print(f"\nFirst 20 labels:")
for label in unique_labels[:20]:
    print(f"  - {label}")

Total unique labels: 13

First 20 labels:
  - ambiance: ambiance
  - ambiance: service
  - app_experience: app_experience
  - app_experience: price
  - cleanliness: ambiance
  - cleanliness: cleanliness
  - delivery: delivery
  - food: food
  - general: general
  - price: app_experience
  - price: price
  - service: cleanliness
  - service: service


In [8]:
print("Converting labels to binary format...")

# Create multi-label binarizer
mlb = MultiLabelBinarizer(classes=unique_labels)

# Transform labels
train_labels = mlb.fit_transform(train_df['label_list'])
val_labels = mlb.transform(val_df['label_list'])

print(f"Train label matrix shape: {train_labels.shape}")
print(f"Val label matrix shape: {val_labels.shape}")
print(f"Number of positive labels in train: {train_labels.sum()}")

Converting labels to binary format...
Train label matrix shape: (1971, 13)
Val label matrix shape: (500, 13)
Number of positive labels in train: 3276


In [9]:
print("Training model...")
print("="*50)

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Transform text to features
print("Converting text to TF-IDF features...")
X_train = vectorizer.fit_transform(train_df['input_text'])
X_val = vectorizer.transform(val_df['input_text'])

print(f"Train features shape: {X_train.shape}")
print(f"Val features shape: {X_val.shape}")

# Train model (One vs Rest with Logistic Regression)
print("\nTraining One-vs-Rest Logistic Regression...")
model = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0, random_state=42))
model.fit(X_train, train_labels)

print("✅ Model training complete!")

Training model...
Converting text to TF-IDF features...


Train features shape: (1971, 6300)
Val features shape: (500, 6300)

Training One-vs-Rest Logistic Regression...
✅ Model training complete!


In [10]:
print("Making predictions on validation set...")
predictions = model.predict(X_val)
print("✅ Predictions complete!")

Making predictions on validation set...
✅ Predictions complete!


In [11]:
print("="*50)
print("EVALUATION RESULTS")
print("="*50)

# Calculate metrics
subset_accuracy = accuracy_score(val_labels, predictions)
f1_macro = f1_score(val_labels, predictions, average='macro', zero_division=0)
f1_micro = f1_score(val_labels, predictions, average='micro', zero_division=0)
f1_weighted = f1_score(val_labels, predictions, average='weighted', zero_division=0)

print(f"\n📊 Overall Metrics:")
print(f"  Subset Accuracy (exact match): {subset_accuracy:.4f}")
print(f"  Macro F1 Score: {f1_macro:.4f}")
print(f"  Micro F1 Score: {f1_micro:.4f}")
print(f"  Weighted F1 Score: {f1_weighted:.4f}")

# Per-label performance
print(f"\n📊 Top 10 Labels by F1 Score:")
precision, recall, f1, _ = precision_recall_fscore_support(val_labels, predictions, average=None, zero_division=0)

label_performance = []
for i, label in enumerate(unique_labels):
    if f1[i] > 0:
        label_performance.append((label, precision[i], recall[i], f1[i]))

label_performance.sort(key=lambda x: x[3], reverse=True)

for i, (label, prec, rec, f1_score) in enumerate(label_performance[:10]):
    print(f"  {i+1}. {label}")
    print(f"     P={prec:.3f}, R={rec:.3f}, F1={f1_score:.3f}")

EVALUATION RESULTS

📊 Overall Metrics:
  Subset Accuracy (exact match): 0.3360
  Macro F1 Score: 0.2364
  Micro F1 Score: 0.5742
  Weighted F1 Score: 0.5138

📊 Top 10 Labels by F1 Score:


NameError: name 'precision_recall_fscore_support' is not defined

In [ ]:
print("="*50)
print("SAMPLE PREDICTIONS")
print("="*50)

# Get predictions as labels
pred_labels = mlb.inverse_transform(predictions)

# Show 10 examples
for i in range(min(10, len(val_df))):
    true_labels = val_df.iloc[i]['label_list']
    pred = pred_labels[i]
    text = val_df.iloc[i]['input_text'][:150]
    
    print(f"\n📝 Example {i+1}:")
    print(f"Text: {text}...")
    print(f"True Labels: {true_labels if true_labels else 'none'}")
    print(f"Pred Labels: {pred if pred else 'none'}")
    
    # Check if match
    if set(true_labels) == set(pred):
        print("✅ MATCH")
    else:
        print("❌ MISMATCH")
    print("-" * 40)

SAMPLE PREDICTIONS

📝 Example 1:
Text: مريم سوتلي الاظافر تحفه اوي ❤️❤️❤️❤️❤️...
True Labels: ['service: service']
Pred Labels: none
❌ MISMATCH
----------------------------------------

📝 Example 2:
Text: التطبيق جميل أتمنى إضافة البحث عن طريق الخريطة وتكون جميع العروض ظاهرة بالخريطة مثل موقع بوكينق وشكرا...
True Labels: ['app_experience: app_experience']
Pred Labels: ('app_experience: app_experience',)
✅ MATCH
----------------------------------------

📝 Example 3:
Text: سراقين مكتوب وصلت السياره والسواق مارضى يقول وين ولمى الرحله من عنده وحطو لي 10 ريال ادفعها حسبي اله ونعم الوكيل فيه...
True Labels: ['service: service', 'delivery: delivery', 'price: price']
Pred Labels: ('service: service',)
❌ MISMATCH
----------------------------------------

📝 Example 4:
Text: سي جيدا...
True Labels: ['general: general']
Pred Labels: none
❌ MISMATCH
----------------------------------------

📝 Example 5:
Text: مكان متاز جدا و الخدمة جيده جدا...
True Labels: ['ambiance: ambiance', 'service: service']

In [ ]:
print("="*50)
print("TEST ON CUSTOM REVIEWS")
print("="*50)

# Custom test reviews
custom_reviews = [
    "The food was amazing and service was quick! I loved it.",
    "Terrible experience, rude staff and cold food. Never coming back.",
    "Nice ambiance but the food was average.",
    "Love this place! Great atmosphere and delicious meals.",
    "The delivery took 2 hours and the food was cold. Very disappointed.",
    "Clean place, friendly staff, will come again!",
    "Worst restaurant ever. Dirty tables and bad service."
]

# Function to predict single review
def predict_single_review(text, vectorizer, model, mlb):
    text_vec = vectorizer.transform([text])
    pred_binary = model.predict(text_vec)
    pred_labels = mlb.inverse_transform(pred_binary)
    return list(pred_labels[0]) if pred_labels[0] else ['none']

# Test all reviews
for i, review in enumerate(custom_reviews, 1):
    predicted = predict_single_review(review, vectorizer, model, mlb)
    print(f"\nReview {i}: {review}")
    print(f"Predicted: {', '.join(predicted)}")
    print("-" * 40)

TEST ON CUSTOM REVIEWS

Review 1: The food was amazing and service was quick! I loved it.
Predicted: service: service
----------------------------------------

Review 2: Terrible experience, rude staff and cold food. Never coming back.
Predicted: none
----------------------------------------

Review 3: Nice ambiance but the food was average.
Predicted: none
----------------------------------------

Review 4: Love this place! Great atmosphere and delicious meals.
Predicted: none
----------------------------------------

Review 5: The delivery took 2 hours and the food was cold. Very disappointed.
Predicted: service: service
----------------------------------------

Review 6: Clean place, friendly staff, will come again!
Predicted: none
----------------------------------------

Review 7: Worst restaurant ever. Dirty tables and bad service.
Predicted: none
----------------------------------------


In [ ]:
# Create a simple function for prediction
def predict_sentiment(text):
    """
    Predict sentiment labels for any text
    """
    text_vec = vectorizer.transform([text])
    pred_binary = model.predict(text_vec)
    pred_labels = mlb.inverse_transform(pred_binary)
    return list(pred_labels[0]) if pred_labels[0] else ['none']

# Example usage
print("="*50)
print("QUICK TEST")
print("="*50)

test_text = "The pizza was delicious and the atmosphere was wonderful!"
result = predict_sentiment(test_text)
print(f"Text: {test_text}")
print(f"Prediction: {result}")

QUICK TEST
Text: The pizza was delicious and the atmosphere was wonderful!
Prediction: ['service: service']
